In [1]:
import os
import numpy as np, pandas as pd

# Locate the dataset directory (adjust if needed)
BASE_PATH = "../input/jigsaw-toxic-comment-classification-challenge"
train_path = os.path.join(BASE_PATH, "train.csv")
test_path = os.path.join(BASE_PATH, "test.csv")

# Load data
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)



In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

# Columns to predict
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Text preprocessing – fill missing comments
train_text = train_df["comment_text"].fillna("").astype(str)
test_text = test_df["comment_text"].fillna("").astype(str)

# TF‑IDF vectorizer (limited features for speed/memory)
vectorizer = TfidfVectorizer(
    max_features=200000,
    ngram_range=(1, 2),
    stop_words="english",
    norm="l2",
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)



In [3]:
# One‑vs‑rest logistic regression (using liblinear for speed)
ovr_clf = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear", max_iter=100, C=4.0, class_weight="balanced", n_jobs=-1
    )
)

# Fit on all toxicity columns
y_train = train_df[label_cols].values
ovr_clf.fit(X_train, y_train)

# Predict probabilities for test set
test_pred = ovr_clf.predict_proba(X_test)



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  

In [4]:
# Build submission DataFrame
submission = pd.DataFrame(test_pred, columns=label_cols)
submission.insert(0, "id", test_df["id"])

# Ensure column order matches the competition format
submission = submission[["id"] + label_cols]

# Write to CSV with the required name
submission.to_csv("submission.csv", index=False)